# 🛣️ Filter Data GPS — Hanya Sekitar Ring Road Utara

Notebook ini memfilter seluruh dataset `all_gps_data_no_dup.parquet` (~169 juta baris)
agar hanya menyisakan titik-titik GPS yang berada **di sekitar Jalan Ring Road Utara**
(Jalan Siliwangi / Jalan Padjajaran — tipe `trunk`), Yogyakarta.

**Pendekatan:**
1. Ambil geometri jalan dari **OSMnx** (dengan cache)
2. Filter hanya jalan tipe **`trunk`** (Ring Road Utara yang sebenarnya)
3. Buat **buffer 750 M** di sekitar jalan
4. Pre-filter cepat via **DuckDB** (bounding box)
5. Filter spasial presisi via **Shapely** (point-in-polygon)
6. **Deduplikasi** hasil (hapus baris duplikat)
7. Simpan hasil ke file **Parquet baru**

> ⚡ Semua operasi berat dijalankan via DuckDB disk-spilling agar aman untuk RAM terbatas.

---
## 0 · Setup & Konfigurasi

In [24]:
import os
from pathlib import Path

import duckdb
import folium
import geopandas as gpd
import numpy as np
import osmnx as ox
import pandas as pd
from shapely.geometry import Point
from shapely.ops import unary_union
from shapely import prepare
from shapely import box

from shared import get_con

# -- Konfigurasi Paths --------------------------------------------------------
INPUT_PARQUET  = "./DataGPS_parquet/all_gps_data_no_dup.parquet"
OUTPUT_PARQUET = "./DataGPS_parquet/gps_ring_road_utara.parquet"
CACHE_DIR      = "./cache"

# Buffer jarak (meter) dari jalan
# Titik GPS dalam radius 750 meter dianggap "di sekitar" Ring Road Utara
BUFFER_METERS = 750

# Batch size untuk proses spasial filter
BATCH_SIZE = 500_000

os.makedirs(CACHE_DIR, exist_ok=True)

# -- DuckDB Connection -------------------------------------------------------
con = get_con(os.getcwd())

# Verifikasi
print("⚙️  DuckDB Configuration:")
for setting in ["memory_limit", "threads", "temp_directory"]:
    val = con.execute(f"SELECT current_setting('{setting}')").fetchone()[0]
    print(f"   {setting}: {val}")

con.execute(f"CREATE OR REPLACE VIEW gps AS SELECT * FROM read_parquet('{INPUT_PARQUET}')")
print(f"\n✅ View 'gps' dibuat dari: {INPUT_PARQUET}")

⚙️  DuckDB Configuration:
   memory_limit: 3.7 GiB
   threads: 4
   temp_directory: /home/repal/Github/skripsi/./.duckdb_tmp

✅ View 'gps' dibuat dari: ./DataGPS_parquet/all_gps_data_no_dup.parquet


---
## 1 · Ambil Geometri Ring Road Utara dari OSMnx

Ring Road Utara Yogyakarta terdiri dari **Jalan Siliwangi** (barat) dan
**Jalan Padjajaran** (timur). Di OSM, jalan ini memiliki 2 tipe:

| Tipe OSM | Keterangan |
|----------|------------|
| `trunk` | **Ring Road Utara yang sebenarnya** (jalan utama) |
| `tertiary` | Jalan layanan / service road paralel |

Kita hanya menggunakan **`trunk`** sesuai gambar Ring Road Utara.

In [25]:
# Cache file untuk semua edges Ring Road Utara
RING_ROAD_CACHE = os.path.join(CACHE_DIR, "ring_road_utara_edges.geojson")
RING_ROAD_FILTERED_CACHE = os.path.join(CACHE_DIR, "ring_road_utara_filtered.geojson")

if os.path.exists(RING_ROAD_CACHE):
    print(f"📂 Memuat dari cache: {RING_ROAD_CACHE}")
    all_ring_edges = gpd.read_file(RING_ROAD_CACHE)
else:
    print("🌐 Mengunduh jaringan jalan dari OSMnx...")

    # Gunakan area yang lebih luas (Sleman) agar Ring Road Utara tercover
    place_name = "Sleman, Daerah Istimewa Yogyakarta, Indonesia"
    G = ox.graph_from_place(place_name, network_type="drive")
    print(f"   Graph: {len(G.nodes)} nodes, {len(G.edges)} edges")

    _, edges = ox.graph_to_gdfs(G)

    # Filter berdasarkan nama jalan Ring Road Utara
    target_names = ["ring road utara", "siliwangi", "padjajaran", "pajajaran"]

    def match_road_name(name_val):
        if isinstance(name_val, str):
            lower = name_val.lower()
            return any(t in lower for t in target_names)
        elif isinstance(name_val, list):
            return any(
                any(t in n.lower() for t in target_names)
                for n in name_val if isinstance(n, str)
            )
        return False

    all_ring_edges = edges[edges["name"].apply(match_road_name)].copy()
    all_ring_edges.to_file(RING_ROAD_CACHE, driver="GeoJSON")
    print(f"   💾 Disimpan ke cache: {RING_ROAD_CACHE}")

print(f"\n📊 Semua segmen Ring Road area: {len(all_ring_edges)}")

# =================================================================
# FILTER TAMBAHAN: hanya bagian UTARA
# Hapus segmen yang turun terlalu jauh ke selatan (sisi barat & timur)
# =================================================================
LAT_THRESHOLD = -7.77  # Batas latitude (selatan)
LONG_THRESHOLD_WEST = 110.342  # Batas longitude (barat)

ring_road_edges = all_ring_edges[
    all_ring_edges["highway"].apply(lambda x: "trunk" in str(x))
].copy()

# Terapkan filter Latitude dan Longitude secara bersamaan
ring_road_edges = ring_road_edges[
    ring_road_edges.geometry.apply(
        lambda geom: (geom.bounds[1] > LAT_THRESHOLD) and (geom.bounds[0] > LONG_THRESHOLD_WEST)
    )
].copy()

print(f"   🔹 Setelah filter utara (lat > {LAT_THRESHOLD}) dan barat (lon > {LONG_THRESHOLD_WEST}): {len(ring_road_edges)} segmen")

# Tampilkan nama jalan yang ditemukan
if "name" in ring_road_edges.columns:
    from collections import Counter
    name_counts = Counter()
    for val in ring_road_edges["name"].dropna():
        if isinstance(val, list):
            for v in val:
                name_counts[str(v)] += 1
        else:
            name_counts[str(val)] += 1
    print(f"   Nama jalan:")
    for n, c in name_counts.most_common():
        print(f"     • {n}: {c} segmen")

# Simpan hasil filter ke GeoJSON
ring_road_edges.to_file(RING_ROAD_FILTERED_CACHE, driver="GeoJSON")
print(f"\n💾 Hasil filter disimpan ke: {RING_ROAD_FILTERED_CACHE}")

📂 Memuat dari cache: ./cache/ring_road_utara_edges.geojson

📊 Semua segmen Ring Road area: 360
   🔹 Setelah filter utara (lat > -7.77) dan barat (lon > 110.342): 98 segmen
   Nama jalan:
     • ['Jalan Padjajaran']: 66 segmen
     • ['Jalan Siliwangi']: 24 segmen
     • ['Jalan Ring Road Utara']: 7 segmen
     • ['Jalan Siliwangi' 'Jalan Padjajaran']: 1 segmen

💾 Hasil filter disimpan ke: ./cache/ring_road_utara_filtered.geojson


---
## 2 · Buat Buffer 1 KM di Sekitar Ring Road Utara

In [26]:
# Proyeksikan ke CRS meter (UTM 49S untuk Yogyakarta) agar buffer dalam meter
ring_road_utm = ring_road_edges.to_crs(epsg=32749)

# Gabungkan semua segmen menjadi satu geometri dan buat buffer
road_union_utm = unary_union(ring_road_utm.geometry)
buffer_utm = road_union_utm.buffer(BUFFER_METERS)

# Konversi kembali ke WGS84 (lat/lon)
buffer_gdf = gpd.GeoDataFrame(geometry=[buffer_utm], crs="EPSG:32749")
buffer_gdf = buffer_gdf.to_crs(epsg=4326)
buffer_polygon = buffer_gdf.geometry.iloc[0]

# Bounding box untuk pre-filter DuckDB
bbox = buffer_polygon.bounds  # (minx, miny, maxx, maxy) = (lon_min, lat_min, lon_max, lat_max)
LON_MIN_BB, LAT_MIN_BB, LON_MAX_BB, LAT_MAX_BB = bbox

print(f"✅ Buffer {BUFFER_METERS}m (1 KM) dibuat di sekitar Ring Road Utara (trunk only)")
print(f"   Bounding Box (WGS84):")
print(f"     Latitude  : {LAT_MIN_BB:.6f} → {LAT_MAX_BB:.6f}")
print(f"     Longitude : {LON_MIN_BB:.6f} → {LON_MAX_BB:.6f}")
print(f"   Luas buffer : {buffer_gdf.to_crs(epsg=32749).area.iloc[0] / 1e6:,.2f} km²")

✅ Buffer 750m (1 KM) dibuat di sekitar Ring Road Utara (trunk only)
   Bounding Box (WGS84):
     Latitude  : -7.776381 → -7.737027
     Longitude : 110.335312 → 110.438228
   Luas buffer : 18.20 km²


---
## 3 · Visualisasi Ring Road Utara + Buffer 750 M (Folium)

In [27]:
# Hitung centroid buffer untuk peta folium
buffer_centroid = buffer_polygon.centroid
m = folium.Map(
    location=[buffer_centroid.y, buffer_centroid.x],
    zoom_start=14,
    tiles="OpenStreetMap",
)

# Plot buffer area
folium.GeoJson(
    buffer_gdf[["geometry"]].to_json(),
    name=f"Buffer {BUFFER_METERS}m",
    style_function=lambda x: {
        "fillColor": "#3498db",
        "color": "#3498db",
        "weight": 1,
        "fillOpacity": 0.15,
    },
).add_to(m)

# Plot jalan Ring Road Utara (trunk only)
folium.GeoJson(
    ring_road_edges[["geometry"]].to_json(),
    name="Ring Road Utara (trunk)",
    style_function=lambda x: {
        "color": "#e74c3c",
        "weight": 4,
        "opacity": 0.9,
    },
).add_to(m)

# Juga plot tertiary untuk perbandingan (abu-abu tipis)
tertiary = all_ring_edges[all_ring_edges["highway"].apply(lambda x: "tertiary" in str(x))]
if len(tertiary) > 0:
    folium.GeoJson(
        tertiary[["geometry"]].to_json(),
        name="Service road (tidak digunakan)",
        style_function=lambda x: {
            "color": "gray",
            "weight": 1,
            "opacity": 0.4,
            "dashArray": "5 5",
        },
    ).add_to(m)

# Layer control
folium.LayerControl().add_to(m)

# Tampilkan peta
m

---
## 4 · Pre-Filter dengan DuckDB (Bounding Box)

Langkah pertama: filter cepat menggunakan bounding box dari buffer 1 KM.
Ini jauh lebih cepat daripada langsung spatial join.

In [28]:
# Hitung total baris asli
total_original = con.execute("SELECT COUNT(*) FROM gps").fetchone()[0]
print(f"📊 Total baris asli: {total_original:,}")

# Hitung baris dalam bounding box
count_bb = con.execute(f"""
    SELECT COUNT(*) FROM gps
    WHERE latitude  BETWEEN {LAT_MIN_BB} AND {LAT_MAX_BB}
      AND longitude BETWEEN {LON_MIN_BB} AND {LON_MAX_BB}
""").fetchone()[0]

print(f"📦 Baris dalam bounding box: {count_bb:,} ({count_bb/total_original*100:.2f}%)")
print(f"   → Reduksi {(1 - count_bb/total_original)*100:.2f}% dari bounding box saja")

📊 Total baris asli: 169,509,324
📦 Baris dalam bounding box: 26,417,319 (15.58%)
   → Reduksi 84.42% dari bounding box saja


---
## 5 · Filter Spasial Presisi + Deduplikasi

Setelah pre-filter bounding box, lakukan pengecekan spasial presisi
menggunakan Shapely `within()` untuk memastikan titik benar-benar dalam
buffer polygon 1 KM.

**Sekaligus menghapus duplikat** — baris dengan (maid, latitude, longitude, timestamp)
yang sama persis dihapus.

In [29]:
# Prepare polygon untuk query spasial yang lebih cepat
prepare(buffer_polygon)

# Query data dalam bounding box secara batch
print(f"🔄 Memulai filter spasial (batch size: {BATCH_SIZE:,})...")
print(f"   Total kandidat (bounding box): {count_bb:,}")
print(f"   Buffer: {BUFFER_METERS}m (1 KM) dari Ring Road Utara (trunk)")
print()

filtered_chunks = []
offset = 0
batch_num = 0
total_kept = 0

while True:
    batch_num += 1

    # Ambil batch dari DuckDB
    df_batch = con.execute(f"""
        SELECT maid, latitude, longitude, timestamp
        FROM gps
        WHERE latitude  BETWEEN {LAT_MIN_BB} AND {LAT_MAX_BB}
          AND longitude BETWEEN {LON_MIN_BB} AND {LON_MAX_BB}
        LIMIT {BATCH_SIZE}
        OFFSET {offset}
    """).df()

    if len(df_batch) == 0:
        break

    # Filter spasial: cek apakah titik berada dalam buffer polygon
    points = gpd.points_from_xy(df_batch["longitude"], df_batch["latitude"])
    mask = gpd.GeoSeries(points).within(buffer_polygon)

    kept = mask.sum()
    total_kept += kept

    if kept > 0:
        filtered_chunks.append(df_batch[mask.values].copy())

    print(f"   Batch {batch_num:>3d} | rows: {len(df_batch):>8,} | "
          f"kept: {kept:>8,} | total kept: {total_kept:>10,} | "
          f"offset: {offset:>10,}")

    offset += BATCH_SIZE

    # Jika batch < BATCH_SIZE, berarti sudah habis
    if len(df_batch) < BATCH_SIZE:
        break

print(f"\n✅ Filter spasial selesai!")
print(f"   Total titik di dalam buffer (sebelum dedup): {total_kept:,}")

🔄 Memulai filter spasial (batch size: 500,000)...
   Total kandidat (bounding box): 26,417,319
   Buffer: 750m (1 KM) dari Ring Road Utara (trunk)

   Batch   1 | rows:  500,000 | kept:  202,665 | total kept:    202,665 | offset:          0
   Batch   2 | rows:  500,000 | kept:  166,498 | total kept:    369,163 | offset:    500,000
   Batch   3 | rows:  500,000 | kept:  176,734 | total kept:    545,897 | offset:  1,000,000
   Batch   4 | rows:  500,000 | kept:  189,981 | total kept:    735,878 | offset:  1,500,000
   Batch   5 | rows:  500,000 | kept:  155,630 | total kept:    891,508 | offset:  2,000,000
   Batch   6 | rows:  500,000 | kept:  159,613 | total kept:  1,051,121 | offset:  2,500,000
   Batch   7 | rows:  500,000 | kept:  161,127 | total kept:  1,212,248 | offset:  3,000,000
   Batch   8 | rows:  500,000 | kept:  172,934 | total kept:  1,385,182 | offset:  3,500,000
   Batch   9 | rows:  500,000 | kept:  174,358 | total kept:  1,559,540 | offset:  4,000,000
   Batch  10 | 

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   Batch  37 | rows:  500,000 | kept:  169,489 | total kept:  6,480,205 | offset: 18,000,000
   Batch  38 | rows:  500,000 | kept:  178,141 | total kept:  6,658,346 | offset: 18,500,000
   Batch  39 | rows:  500,000 | kept:  167,268 | total kept:  6,825,614 | offset: 19,000,000
   Batch  40 | rows:  500,000 | kept:  141,494 | total kept:  6,967,108 | offset: 19,500,000
   Batch  41 | rows:  500,000 | kept:  190,410 | total kept:  7,157,518 | offset: 20,000,000
   Batch  42 | rows:  500,000 | kept:  168,966 | total kept:  7,326,484 | offset: 20,500,000
   Batch  43 | rows:  500,000 | kept:  172,303 | total kept:  7,498,787 | offset: 21,000,000
   Batch  44 | rows:  500,000 | kept:  175,593 | total kept:  7,674,380 | offset: 21,500,000


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   Batch  45 | rows:  500,000 | kept:  178,752 | total kept:  7,853,132 | offset: 22,000,000


...

   Batch  48 | rows:  500,000 | kept:  183,817 | total kept:  8,379,445 | offset: 23,500,000
   Batch  49 | rows:  500,000 | kept:  178,892 | total kept:  8,558,337 | offset: 24,000,000


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   Batch  50 | rows:  500,000 | kept:  152,351 | total kept:  8,710,688 | offset: 24,500,000
   Batch  51 | rows:  500,000 | kept:  129,862 | total kept:  8,840,550 | offset: 25,000,000
   Batch  52 | rows:  500,000 | kept:  173,383 | total kept:  9,013,933 | offset: 25,500,000


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   Batch  53 | rows:  417,319 | kept:  110,097 | total kept:  9,124,030 | offset: 26,000,000

✅ Filter spasial selesai!
   Total titik di dalam buffer (sebelum dedup): 9,124,030


---
## 6 · Gabungkan, Deduplikasi & Periksa Hasil

In [30]:
if len(filtered_chunks) == 0:
    print("⚠️  Tidak ada data yang ditemukan dalam buffer!")
    df_result = pd.DataFrame(columns=["maid", "latitude", "longitude", "timestamp"])
else:
    df_result = pd.concat(filtered_chunks, ignore_index=True)

# === DEDUPLIKASI ===
n_before_dedup = len(df_result)
df_result = df_result.drop_duplicates(
    subset=["maid", "latitude", "longitude", "timestamp"],
    keep="first"
).reset_index(drop=True)
n_after_dedup = len(df_result)
n_dupes_removed = n_before_dedup - n_after_dedup

print(f"📊 Ringkasan Hasil Filter:")
print(f"   Total baris asli          : {total_original:>14,}")
print(f"   Setelah bounding box      : {count_bb:>14,} ({count_bb/total_original*100:.2f}%)")
print(f"   Setelah filter spasial    : {n_before_dedup:>14,} ({n_before_dedup/total_original*100:.4f}%)")
print(f"   Duplikat dihapus          : {n_dupes_removed:>14,}")
print(f"   ✅ Hasil akhir (bersih)   : {n_after_dedup:>14,} ({n_after_dedup/total_original*100:.4f}%)")
print(f"   Jumlah MAID unik          : {df_result['maid'].nunique():>14,}")
print()
print("Preview data:")
df_result.head(10)

📊 Ringkasan Hasil Filter:
   Total baris asli          :    169,509,324
   Setelah bounding box      :     26,417,319 (15.58%)
   Setelah filter spasial    :      9,124,030 (5.3826%)
   Duplikat dihapus          :        160,459
   ✅ Hasil akhir (bersih)   :      8,963,571 (5.2880%)
   Jumlah MAID unik          :        400,643

Preview data:


,maid,latitude,longitude,timestamp
0,002fbd9b-217c-43d7-b003-5d182d5bf427,-7.760000,110.400002,1644982958
1,002fbd9b-217c-43d7-b003-5d182d5bf427,-7.749632,110.379005,1645230753
2,002fbd9b-217c-43d7-b003-5d182d5bf427,-7.749632,110.379005,1645230753
3,002fbd9b-217c-43d7-b003-5d182d5bf427,-7.749640,110.378998,1645230753
4,002fbd9b-217c-43d7-b003-5d182d5bf427,-7.749632,110.379005,1645230814
5,002fbd9b-217c-43d7-b003-5d182d5bf427,-7.749632,110.379005,1645230814
6,002fec53-2f7f-4eb6-a4c7-eaeccde48bbd,-7.760000,110.400002,1643693474
7,002fec53-2f7f-4eb6-a4c7-eaeccde48bbd,-7.760000,110.400002,1643701015
8,002fec53-2f7f-4eb6-a4c7-eaeccde48bbd,-7.760000,110.400002,1643703462
9,002fec53-2f7f-4eb6-a4c7-eaeccde48bbd,-7.760000,110.400002,1643706649


In [31]:
# Statistik deskriptif
print("📈 Statistik Deskriptif:")
print(df_result.describe())

📈 Statistik Deskriptif:
           latitude     longitude     timestamp
count  8.963571e+06  8.963571e+06  8.963571e+06
mean  -7.757429e+00  1.103912e+02  1.643423e+09
std    7.345667e-03  2.368786e-02  6.397585e+06
min   -7.776373e+00  1.103353e+02  1.634947e+09
25%   -7.763360e+00  1.103732e+02  1.637652e+09
50%   -7.759407e+00  1.103905e+02  1.642675e+09
75%   -7.751479e+00  1.104104e+02  1.647279e+09
max   -7.737102e+00  1.104382e+02  1.654604e+09


---
## 7 · Visualisasi Hasil Filter (Folium)

In [32]:
# Buat peta interaktif dengan folium
buffer_centroid = buffer_polygon.centroid
m2 = folium.Map(
    location=[buffer_centroid.y, buffer_centroid.x],
    zoom_start=14,
    tiles="OpenStreetMap",
)

# Buffer area
folium.GeoJson(
    buffer_gdf[["geometry"]].to_json(),
    name=f"Buffer {BUFFER_METERS}m",
    style_function=lambda x: {
        "fillColor": "#3498db",
        "color": "#3498db",
        "weight": 1,
        "fillOpacity": 0.1,
    },
).add_to(m2)

# Ring Road Utara (trunk only)
folium.GeoJson(
    ring_road_edges[["geometry"]].to_json(),
    name="Ring Road Utara (trunk)",
    style_function=lambda x: {
        "color": "#e74c3c",
        "weight": 3,
        "opacity": 0.9,
    },
).add_to(m2)

# Sampel titik GPS (max 5000 untuk performa folium)
from folium.plugins import FastMarkerCluster

sample_size = min(5_000, len(df_result))
if sample_size > 0:
    sample = df_result.sample(n=sample_size, random_state=42)
    # Gunakan CircleMarker untuk titik GPS
    gps_group = folium.FeatureGroup(name=f"GPS Points (sample {sample_size:,})")
    for _, row in sample.iterrows():
        folium.CircleMarker(
            location=[row["latitude"], row["longitude"]],
            radius=2,
            color="#2ecc71",
            fill=True,
            fill_color="#2ecc71",
            fill_opacity=0.5,
            weight=1,
        ).add_to(gps_group)
    gps_group.add_to(m2)

# Layer control
folium.LayerControl().add_to(m2)

print(f"📊 Total: {len(df_result):,} titik, {df_result['maid'].nunique():,} MAID unik")

# Tampilkan peta
m2

📊 Total: 8,963,571 titik, 400,643 MAID unik


---
## 8 · Simpan ke Parquet Baru

In [33]:
# Simpan menggunakan DuckDB (lebih efisien untuk kompresi)
con.execute(f"""
    COPY (
        SELECT DISTINCT maid, latitude, longitude, timestamp
        FROM df_result
        ORDER BY maid, timestamp
    ) TO '{OUTPUT_PARQUET}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

# Verifikasi file
out_path = Path(OUTPUT_PARQUET)
out_size_mb = out_path.stat().st_size / 1024 / 1024

# Verifikasi jumlah baris
verify_count = con.execute(f"SELECT COUNT(*) FROM read_parquet('{OUTPUT_PARQUET}')").fetchone()[0]

# Verifikasi tidak ada duplikat
verify_distinct = con.execute(f"""
    SELECT COUNT(*) FROM (
        SELECT DISTINCT maid, latitude, longitude, timestamp
        FROM read_parquet('{OUTPUT_PARQUET}')
    )
""").fetchone()[0]

print(f"✅ Data berhasil disimpan!")
print(f"   📁 Path       : {out_path.resolve()}")
print(f"   📏 Ukuran     : {out_size_mb:.2f} MB")
print(f"   📊 Baris      : {verify_count:,}")
print(f"   🔍 Distinct   : {verify_distinct:,} (harus sama dengan baris)")
print(f"   🔽 Reduksi    : {(1 - verify_count/total_original)*100:.2f}% dari dataset asli")
print(f"   ✅ Duplikat   : {'TIDAK ADA ✓' if verify_count == verify_distinct else '⚠️ MASIH ADA!'}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

✅ Data berhasil disimpan!
   📁 Path       : /home/repal/Github/skripsi/DataGPS_parquet/gps_ring_road_utara.parquet
   📏 Ukuran     : 61.16 MB
   📊 Baris      : 8,963,571
   🔍 Distinct   : 8,963,571 (harus sama dengan baris)
   🔽 Reduksi    : 94.71% dari dataset asli
   ✅ Duplikat   : TIDAK ADA ✓


---
## 9 · Verifikasi Data Output

In [34]:
# Baca kembali dan tampilkan info
con.execute(f"CREATE OR REPLACE VIEW gps_rru AS SELECT * FROM read_parquet('{OUTPUT_PARQUET}')")

print("=" * 70)
print("VERIFIKASI DATA OUTPUT — gps_ring_road_utara.parquet")
print("=" * 70)

# Schema
print("\n📋 Schema:")
schema = con.execute("""
    SELECT column_name AS "Kolom", data_type AS "Tipe"
    FROM information_schema.columns
    WHERE table_name = 'gps_rru'
    ORDER BY ordinal_position
""").df()
print(schema.to_string(index=False))

# Dimensi
dims = con.execute("""
    SELECT
        COUNT(*)             AS total_rows,
        COUNT(DISTINCT maid) AS unique_maids,
        MIN(latitude)        AS lat_min,
        MAX(latitude)        AS lat_max,
        MIN(longitude)       AS lon_min,
        MAX(longitude)       AS lon_max,
        MIN(timestamp)       AS ts_min,
        MAX(timestamp)       AS ts_max
    FROM gps_rru
""").df()

print(f"\n📊 Dimensi:")
print(f"   Total baris   : {int(dims['total_rows'].iloc[0]):,}")
print(f"   MAID unik     : {int(dims['unique_maids'].iloc[0]):,}")
print(f"   Latitude      : {dims['lat_min'].iloc[0]:.6f} → {dims['lat_max'].iloc[0]:.6f}")
print(f"   Longitude     : {dims['lon_min'].iloc[0]:.6f} → {dims['lon_max'].iloc[0]:.6f}")

import datetime
ts_min = datetime.datetime.fromtimestamp(int(dims['ts_min'].iloc[0]))
ts_max = datetime.datetime.fromtimestamp(int(dims['ts_max'].iloc[0]))
print(f"   Timestamp     : {ts_min} → {ts_max}")

# Preview
print("\n📝 Preview (5 baris pertama):")
con.execute("SELECT * FROM gps_rru LIMIT 5").df()

VERIFIKASI DATA OUTPUT — gps_ring_road_utara.parquet

📋 Schema:
    Kolom    Tipe
     maid VARCHAR
 latitude  DOUBLE
longitude  DOUBLE
timestamp  BIGINT

📊 Dimensi:
   Total baris   : 8,963,571
   MAID unik     : 400,643
   Latitude      : -7.776373 → -7.737102
   Longitude     : 110.335327 → 110.438220
   Timestamp     : 2021-10-23 07:00:00 → 2022-06-07 19:11:45

📝 Preview (5 baris pertama):


,maid,latitude,longitude,timestamp
0,00000000-0000-0000-0000-000000000000,-7.759640,110.399150,1653797201
1,00001a8b-69eb-4f44-809c-843b584f9797,-7.763116,110.433411,1639612448
2,00001a8b-69eb-4f44-809c-843b584f9797,-7.763116,110.433410,1639692911
3,00001a8b-69eb-4f44-809c-843b584f9797,-7.763116,110.433411,1639709015
4,00001a8b-69eb-4f44-809c-843b584f9797,-7.763117,110.433390,1639808570


In [35]:
# Cleanup
con.close()
print("\n🏁 Selesai! Koneksi DuckDB ditutup.")
print(f"   File output: {Path(OUTPUT_PARQUET).resolve()}")


🏁 Selesai! Koneksi DuckDB ditutup.
   File output: /home/repal/Github/skripsi/DataGPS_parquet/gps_ring_road_utara.parquet
